# bottleneck-latent-projection — worked example 3: Full bottleneck encode then decode round trip on a conv feature map

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bottleneck-latent-projection`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A full bottleneck round trip chains the encoder (flatten -> Linear -> ReLU -> Linear) and the decoder mirror (Linear -> ReLU -> Linear -> un-flatten). The reconstruction has the same shape as the original feature map, but generally different values because the latent dimension is much smaller than `C*H*W` - the bottleneck forces information loss.

## Worked solution

**Step 1 - encode.** Flatten `(B, C, H, W)` to `(B, C*H*W)`, apply `Linear(in, hidden)` + ReLU, then `Linear(hidden, latent)` with no activation. This produces the compressed code `z` of shape `(B, latent)`.

**Step 2 - decode.** Apply `Linear(latent, hidden)` + ReLU, then `Linear(hidden, in)` with no activation to get the flattened reconstruction, then un-flatten with `rearrange('b (c h w) -> b c h w', ...)`.

**Step 3 - why shapes match but values don't.** The decoder's final Linear outputs exactly `C*H*W` features, so reshaping recovers `(B, C, H, W)`. But because `latent (=3)` is far smaller than `in (=128)`, the encoder cannot preserve every detail - the round trip is lossy by design. We assert the shape matches and report the reconstruction error.

**Step 4 - check.** We run a `(B=4, 8, 4, 4)` batch through both halves, print that the reconstruction shape equals the input shape, and print the mean squared reconstruction error to make the lossy nature concrete.

In [ ]:
def roundtrip(x, We1, be1, We2, be2, Wd1, bd1, Wd2, bd2, C, H, W):
    flat = rearrange(x, 'b c h w -> b (c h w)')
    z = t.relu(flat @ We1.T + be1) @ We2.T + be2
    h = t.relu(z @ Wd1.T + bd1)
    rec_flat = h @ Wd2.T + bd2
    rec = rearrange(rec_flat, 'b (c h w) -> b c h w', c=C, h=H, w=W)
    return z, rec

t.manual_seed(0)
B, C, H, W = 4, 8, 4, 4
in_f, hidden, latent = C * H * W, 32, 3
x = t.randn(B, C, H, W)
We1 = t.randn(hidden, in_f); be1 = t.randn(hidden)
We2 = t.randn(latent, hidden); be2 = t.randn(latent)
Wd1 = t.randn(hidden, latent); bd1 = t.randn(hidden)
Wd2 = t.randn(in_f, hidden); bd2 = t.randn(in_f)
z, rec = roundtrip(x, We1, be1, We2, be2, Wd1, bd1, Wd2, bd2, C, H, W)
print('latent shape:', tuple(z.shape))
print('rec matches input shape:', rec.shape == x.shape)
print('mse:', round(((rec - x) ** 2).mean().item(), 4))